# 示例 1：环形槽

以下面的示例网络（来自 scikit-rf `data` 文件夹中的 *环形槽*）演示了矢量拟合法。可以在 [矢量拟合教程](../../tutorials/VectorFitting.ipynb) 中找到更多说明和背景信息。

In [ ]:
import matplotlib.pyplot as mplt
import numpy as np

import skrf
from skrf.vectorFitting import VectorFitting

要创建一个 `VectorFitting` 实例，需要传递一个包含 N 端口频率响应的 `Network` 对象。在此示例中，使用了 *环形槽*，可以直接从 scikit-rf 加载为 `Network` 对象：

In [ ]:
nw = skrf.data.ring_slot
vf = VectorFitting(nw)

现在，可以执行矢量拟合。需要指定极点的数量，这取决于响应的*特性*。平滑的响应只需要很少的极点（2-5个）。在这种情况下，3个实极点就足够了：

In [ ]:
vf.vector_fit(n_poles_real=3, n_poles_cmplx=0)

如日志输出所示（此处未显示），极点移动过程在仅进行 5 次迭代步骤后便迅速收敛。这也可以通过查看收敛曲线来验证：

In [ ]:
vf.plot_convergence()

拟合后的模型参数现在存储在类属性 `poles`、`residues`、`proportional_coeff` 和 `constant_coeff` 中，以便后续使用。为了验证结果，可以将模型响应与原始网络响应进行比较。一种方法是分析均方根误差的大小，对于良好的拟合，该值应该小于 0.05：

In [ ]:
vf.get_rms_error()

由于该模型会在任何给定的频率下返回结果，因此有必要手动检查其在原始样本频率范围之外的响应，方法是在较低和较高的频率下绘制响应曲线：

In [ ]:
freqs1 = np.linspace(0, 200e9, 201)
fig, ax = mplt.subplots(2, 2)
fig.set_size_inches(12, 8)
vf.plot_s_mag(0, 0, freqs1, ax=ax[0][0]) # plot s11
vf.plot_s_mag(1, 0, freqs1, ax=ax[1][0]) # plot s21
vf.plot_s_mag(0, 1, freqs1, ax=ax[0][1]) # plot s12
vf.plot_s_mag(1, 1, freqs1, ax=ax[1][1]) # plot s22
fig.tight_layout()
mplt.show()

为了在电路仿真中使用该模型，可以基于拟合参数创建一个等效电路。目前，这仅针对 SPICE 进行了实现，但等效电路的结构可以应用于任何类型的电路仿真器。请注意：已打印关于非被动拟合的 UserWarning（参见上面的 vector_fit() 输出）。对模型被动性的评估和强制执行在 [此示例](./vectorfitting_ex4_passivity.ipynb) 中进行了更详细的描述，这对于电路仿真器中的某些应用场景（例如，瞬态仿真）非常重要。

`vf1.write_spice_subcircuit_s('/home/vinc/Desktop/ring_slot.sp')`

为了快速测试，该子电路包含在 [QUCS-S](https://ra3xdh.github.io/) 的原理图中，用于进行交流仿真和基于端口电压和电流的散射参数计算（参见方程）：<img src="./ngspice_ringslot_schematic.svg" />来自 [ngspice](http://ngspice.sourceforge.net/) 的仿真输出与上面的图表结果相符：<img src="./ngspice_ringslot_sp_mag.svg" /><img src="./ngspice_ringslot_sp_smith.svg" />